Импорт библиотек и устройство

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import models
from torchvision.transforms import functional
from torchvision.transforms import InterpolationMode
from torchvision import transforms
import json
import os
import math
import time
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

device: cpu
torch: 2.10.0+cpu
torchvision: 0.25.0+cpu


Загрузка датасета

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# For visualization (without normalization)
visual_transform = transforms.Compose([
    transforms.ToTensor()
])


print("Loading STL10 dataset...")
full_train_dataset = torchvision.datasets.STL10(
    root='./data',  # Standard relative path
    split='train',  # Training split (5000 images)
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.STL10(
    root='./data',
    split='test',  # Test split (8000 images)
    download=True,
    transform=transform
)

# Create validation split from training data (80/20 split)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"\nDataset splits:")
print(f"  Train: {len(train_dataset)} images")
print(f"  Validation: {len(val_dataset)} images")
print(f"  Test: {len(test_dataset)} images")
print(f"  Total images: {len(train_dataset) + len(val_dataset) + len(test_dataset)}")
print(f"  Number of classes: 10")

Loading STL10 dataset...


 41%|████▏     | 1.09G/2.64G [29:15<41:18, 624kB/s]    


KeyboardInterrupt: 

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
vis_dataset = torchvision.datasets.STL10(
    root='./data',
    split='train',
    download=False,  # Already downloaded
    transform=visual_transform
)

def denormalize(image, mean, std):
    """Denormalize image for visualization"""
    image = image.clone()
    for t, m, s in zip(image, mean, std):
        t.mul_(s).add_(m)
    return image

# Get class names for STL10
class_names = [
    'airplane', 'bird', 'car', 'cat', 'deer', 
    'dog', 'horse', 'monkey', 'ship', 'truck'
]

# Display sample images
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
axes = axes.ravel()

# Show 15 random samples from training set
for idx in range(15):
    # Get random image
    random_idx = np.random.randint(0, len(vis_dataset))
    image, label = vis_dataset[random_idx]
    
    # Denormalize if needed (but visual_transform doesn't normalize)
    if image.shape[0] == 3:
        # Image is already in [0,1] range
        img_display = image.permute(1, 2, 0).numpy()
    else:
        img_display = image
    
    axes[idx].imshow(img_display)
    axes[idx].set_title(f'Class: {class_names[label]}')
    axes[idx].axis('off')

plt.suptitle('STL10 Sample Images (96x96 pixels)', fontsize=14)
plt.tight_layout()
plt.show()